# MapBiomas Fire Monitor — Pipeline (M0-M6)

## Documentation and User Guide
Expand this section to understand the data architecture, environment requirements, and workflow.

#### Introduction and Context

This pipeline is the automated processing core for the **MapBiomas Fire Scar Monitoring**. It is organized into sequential stages (M0-M8), each with a specific responsibility:

* **M0 — Setup and Authentication**: Colab-native setup. Clones the repository, installs dependencies, configures GCS and GEE project paths via `set_global_opts()`, authenticates with Earth Engine. Centralizes all configuration (country, campaign, sensor, periodicity, GEE/GCS prefixes).

* **M1 — Export (GEE → GCS)**: Multi-sensor satellite image export. Handles Sentinel-2 (and in future: Landsat 5/7/8/9, MODIS, HLS). Applies sensor-specific radiometric corrections, cloud masking (QA_PIXEL, Fmask, CS), and exports optimized GeoTIFF chunks to Google Cloud Storage. Supports monthly and annual periods with configurable mosaics (minnbr, minnbr_buffer).

* **M2 — Mosaic Assembly (COG)**: Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT for virtual stacking. Produces compressed, tiled COGs with LZW compression, ready for analysis and model inference.

* **M3 — Sample Collection (GEE Toolkit JavaScript)**: Training data collection via a custom GEE JavaScript Toolkit. Users draw fire (burned) and notFire (unburned) polygons on satellite imagery. Samples are exported with full metadata. Supports multi-country, multi-language (EN/ES/PT/FR/ID), and regional classification layers (classification, probability, dayOfYear).

* **M4 — DNN Training**: Deep Neural Network classification. Uses a flexible band extraction matrix to sample pixel values from M2 mosaics using M3 polygon samples. Trains a DNN classifier with configurable architecture, generates t-SNE audit plots, and saves model weights and metadata to GCS.

* **M5 — Tile Classification**: Produces int16 2-band COG tile rasters (probability 0–1000 + dayOfYear) for each cim-world grid cell. Classifies tiles independently using a trained DNN model from M4. Results are consumed by M6 for mosaicking and publication.

* **M6 — Mosaic, Statistics & Publication**: Assembles classified M5 tiles into regional mosaics, computes burned area statistics (CSV), and uploads results to Google Earth Engine. Includes an interactive dashboard for batch monitoring, analytics filtering, and coverage visualization.

* **M7 — Post-Classification Masks**: 🔨 Under development. Will apply spatial and temporal masks to classified outputs.

* **M8 — Final Version Evaluation & Official Release**: 🔨 Under development. Will evaluate candidate versions and select the official release for publication.

**Global Inputs:** Raw image collections from Google Earth Engine (Sentinel-2), vector polygons for training (samples), and auxiliary land cover maps.

**Global Outputs:** Optimized chunks and mosaics (COGs) on Google Cloud Storage (GCS), neural network model weights (DNN), int16 2-band classification rasters (probability + dayOfYear), consolidated statistics (CSV), and versioned ImageCollections on GEE.

> **Note:** Both the local PC disk and Colab temporary storage act as **ephemeral space**. Persistence always occurs in the **Google Cloud Storage (GCS) Bucket**.


#### Data Lifecycle and Naming Rules

| Stage | Weight | Inputs → Outputs | Naming Rule | Example |
| :--- | :--- | :--- | :--- | :--- |
| **M1: Export** | **Light** | **IN:** GEE Collections<br>**OUT:** GCS Chunks | `image_{country}_fire_{sensor}_{mosaic}_{band}_{temporal_id}_{suffix}` | `image_peru_fire_sentinel2_minnbr_blue_2025_08_00000-00000` |
| **M2: Mosaic** | **Medium** | **IN:** GCS Chunks<br>**OUT:** COG Mosaics (GCS) | same as M1 seed, stored in `/COG/` dir | `image_peru_fire_sentinel2_minnbr_blue_2025_08` |
| **M3: Samples** | **Light** | **IN:** Mosaics (M2)<br>**OUT:** Polygons (GCS/Asset) | `sample_{id}_{temporal_id}`| `sample_0001_2025_07` |
| **M4: Train** | **Medium** | **IN:** M3 Samples + M2 Mosaics<br>**OUT:** DNN (GCS) | `training_{id}_{shortname}_{sensor}` | `training_0001_amazon_sentinel2` |
| **M5: Classify**| **Heavy**| **IN:** M4 Model + M2 Mosaics<br>**OUT:** int16 Tiles (GCS) | `tile_{region}_{period}_{cell}.tif` | `tile_amazonia_2025_08_SA-24-V-C.tif` |
| **M6: Publish**| **Heavy**| **IN:** M5 Tiles<br>**OUT:** Mosaics + Stats CSV + GEE Assets | `{region}_{period}_mosaic.tif` | `peru_2025_08_mosaic.tif` |
| **M7: Mask** | — | Post-Classification Masks | — | 🔨 Under development |
| **M8: Release**| — | Final Version Evaluation & Release | — | 🔨 Under development |


### Data Architecture and Relationships (M1-M7)

#### Persistence Map (Where to find the data)

| Stage | Extension | Main Cloud Storage Path (GCS) |
| :--- | :--- | :--- |
| **M1: Export** | `.tif` | `{gcs_library_images_prefix}/LIBRARY_IMAGES/{SENSOR}/MONTHLY/{MOSAIC}/{date}/CHUNKS/` |
| **M2: Mosaic** | `.tif` | `{gcs_library_images_prefix}/LIBRARY_IMAGES/{SENSOR}/MONTHLY/{MOSAIC}/{date}/COG/` |
| **M3: Samples** | `.csv` | `{gcs_campaigns_prefix}/{campaign}/LIBRARY_SAMPLES/` |
| **M4: Train** | `.pb / .json` | `{gcs_campaigns_prefix}/{campaign}/LIBRARY_MODELS/training_{id}_{shortname}_{sensor}/` |
| **M5: Classify** | `.tif` | `{gcs_campaigns_prefix}/{campaign}/LIBRARY_CLASSIFICATIONS/{model_id}/CLASSIFIED_TILES/` (tiles) |
| | `.tif` | `{gcs_campaigns_prefix}/{campaign}/LIBRARY_CLASSIFICATIONS/{model_id}/CLASSIFIED_REGION/` (mosaics) |
| | `.csv` | `{gcs_campaigns_prefix}/{campaign}/LIBRARY_CLASSIFICATIONS/{model_id}/STATS/` (statistics) |

## [M0] — Environment Setup
Clones the repository (first run only), installs system and Python dependencies (GDAL, gcsfs, rasterio, earthengine-api), configures GCS bucket and GEE project via `set_global_opts()`, authenticates with Earth Engine, and registers the region FeatureCollection.

In [ ]:
# M0 -- Colab Setup
import sys, os, subprocess

# -- Clone repository (first run only) --
if not os.path.exists("fire_monitor"):
    subprocess.run(["git", "clone", "https://github.com/mapbiomas/peru-fire.git", "fire_monitor"])

# -- Install system dependencies --
subprocess.run(["apt-get", "update", "-qq"])
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"])
subprocess.run(["pip", "install", "-q", "earthengine-api", "gcsfs", "rasterio", "scipy", "tqdm"])

# -- Add to Python path --
repo_path = "/content/fire_monitor/mapbiomas_fire_monitor/version_01/src/core"
if repo_path not in sys.path: sys.path.insert(0, repo_path)
os.chdir(repo_path)

# -- Configure project --
from M0_auth_config import set_global_opts, authenticate, print_config

set_global_opts(
    # === Active Campaign ===
    country='peru',                                         # country/region code (e.g. 'peru', 'bolivia')
    campaign='MONITOR_01',                                  # campaign (e.g. 'MONITOR_01', 'COLLECTION_01', 'COLLECTION_02')

    # === Google Cloud Storage ===
    gcs_bucket='mapbiomas-fire',                            # bucket name
    gcs_library_images_prefix='sudamerica/peru/CATALOG_01', # prefix for LIBRARY_IMAGES
    gcs_campaigns_prefix='sudamerica/peru/CATALOG_01',      # prefix for campaign outputs

    # === Google Earth Engine ===
    gee_project='mapbiomas-peru',                            # GEE Cloud Project ID
    gee_library_images_prefix='projects/mapbiomas-peru/assets/FIRE/CATALOG_01',  # asset path for images
    gee_campaigns_prefix='projects/mapbiomas-peru/assets/FIRE/CATALOG_01',       # asset path for campaigns
    asset_regions='projects/mapbiomas-peru/assets/FIRE/CATALOG_01/MONITOR_01/AUXILIARY/regiones_fuego_peru_v1',

    # === Pipeline Settings ===
    sensor=['sentinel2'],                                   # list: 'sentinel2', 'landsat', 'hls', 'modis'
    periodicity=['monthly'],                                 # list: 'monthly', 'yearly'
    mosaic_methods=['minnbr', 'minnbr_buffer'],              # list: 'minnbr', 'minnbr_buffer', 'median', 'minndvi'
    personal_task_flag='MONITOR',                            # prefix for GEE task names
    language='en',                                           # 'en', 'es', 'pt'
)

authenticate()
print_config()

## [M1] — Export (GEE → GCS)
Multi-sensor satellite image export from GEE ImageCollections to GCS. Handles Sentinel-2 (and in future: Landsat (5/7/8/9), MODIS, and HLS) with sensor-specific radiometric corrections, cloud masking, and configurable mosaic types (MINNBR, MINNBR_BUFFER, etc.).

In [ ]:
from M1_export_ui import run_ui, start_export

ui_exporter = run_ui(years=[2025,2026])

In [ ]:
# Export trigger (GEE -> GCS)

start_export(ui_exporter)

## [M2] — Mosaic Assembly (COG)

Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT virtual stacking. Produces compressed, tiled COGs with DEFLATE compression.

In [ ]:
from M2_mosaic_ui import run_ui, start_mosaic_assembly
ui_assembler = run_ui(years=[2025, 2026])

In [ ]:
# Mosaic trigger (Local/GCS COG)
start_mosaic_assembly(ui_assembler)

## [M3] — Sample Collection (GEE Toolkit)
Training data collection via a custom GEE JavaScript Toolkit. Users draw fire/notFire polygons on satellite imagery. Samples are exported to GEE Assets and GCS with full metadata (satellite, date, region, campaign).

In [ ]:
from M3_sample_ui import show_toolkit_links
show_toolkit_links()

## [M4] — DNN Training
Trains a Deep Neural Network classifier using a flexible band extraction matrix. Samples pixels from M2 mosaics using M3 polygons, trains a configurable DNN, generates t-SNE audit plots, and saves model weights and metadata to GCS.

In [ ]:
from M4_model_trainer import run_ui, start_training
ui_trainer = run_ui()

In [ ]:
# Training trigger
start_training(ui_trainer)

## [M5] — Tile-Based Classification
Tile-based burned area classification using a trained DNN model. Loads the model from GCS, retrieves the cell grid for the target region, classifies each tile independently, and generates per-tile classified rasters with local statistics. Results are consumed by M6 for mosaicking and publication.

In [ ]:
# 1. M5 Scheduling Interface
from M5_classifier_ui import run_m5_ui
ui_m5 = run_m5_ui(years=[2025, 2026], periodicity_active=["monthly"])

In [ ]:
# 2. M5 Async Processing Engine
# n_workers: auto (cpu_count). Force com: run_m5_workplan(n_workers=4, progress_callback=ui_m5._on_tile_progress)
from M5_classifier import run_m5_workplan
run_m5_workplan(progress_callback=ui_m5._on_tile_progress)

## [M6] — Mosaic, Statistics & Publication
Assembles classified tiles into regional mosaics, computes burned area statistics, and publishes results to Google Earth Engine as ImageCollections. Provides an interactive dashboard for monitoring batch progress, analytics, and coverage.

In [ ]:
# 3. M6 Scheduling Interface
from M6_ui import run_m6_ui
ui_m6 = run_m6_ui()

In [ ]:
# 2. M6 Publish (mosaic + stats + GEE upload)
# Runs only when classified tiles (COMPLETED) are available in GCS.
# upload_gee=True sends regional mosaics to Google Earth Engine.
from M6_publisher import run_m6_publish
run_m6_publish(upload_gee=True)

## [M7] — Post-Classification Masks
🔨 **Under development.** Will apply spatial and temporal masks to classified outputs before final publication.

In [ ]:
# 🔨 M7 — Post-Classification Masks (Under development)
# Apply spatial and temporal masks to classified outputs.
# from M7_masks import run_m7_masks

## [M8] — Final Version Evaluation & Official Release
🔨 **Under development.** Will evaluate candidate versions and select the official release for publication.

In [ ]:
# 🔨 M8 — Final Version Evaluation & Official Release (Under development)
# Evaluate candidate versions and select the official release.
# from M8_release import run_m8_release